# Convergence vs path count

European call Monte Carlo price against the Black–Scholes value as `n` grows.
Control variates are off so the error is visible.

In [ ]:
import matplotlib.pyplot as plt

from monte_carlo_option_engine import (
    Contract,
    ContractKind,
    Market,
    black_scholes,
    price_mc,
)

market = Market(S=100, T=0.5, r=0.04, q=0.01, sigma=0.25)
call = Contract(K=105, kind=ContractKind.euro_call)
bs = black_scholes(market, call)
ns = [500, 1_000, 2_000, 4_000, 8_000]
prices = []
half_widths = []
for n in ns:
    result = price_mc(
        market,
        call,
        trial_count=n,
        seed=0,
        antithetic=False,
        control_variate=False,
    )
    prices.append(result.price)
    half_widths.append(1.96 * result.stderr)
    print(f"n={n:5d}  price={result.price:.4f}  stderr={result.stderr:.4f}")

fig, ax = plt.subplots()
ax.errorbar(ns, prices, yerr=half_widths, fmt="o-", capsize=4, label="MC ± 95% CI")
ax.axhline(bs, color="C1", linestyle="--", label=f"BS {bs:.4f}")
ax.set_xlabel("Paths")
ax.set_ylabel("Price")
ax.set_title("European call: MC vs Black–Scholes")
ax.legend()
fig